# Tree Pricing Demo

Simple CRR binomial pricing for European, American, and Bermudan options.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
from statistics import NormalDist

from trees.crr import price_option

FIGURES = ROOT / "figures"
FIGURES.mkdir(exist_ok=True)


In [ ]:
price = price_option(
    S=100.0, K=100.0, r=0.05, sigma=0.20, T=1.0, N=3,
    kind="call", style="european",
)
print(f"European call (N=3): {price:.4f}")


In [ ]:
args = dict(S=100.0, K=100.0, r=0.05, sigma=0.20, T=1.0, N=30, kind="put")

euro_put = price_option(**args, style="european")
amer_put = price_option(**args, style="american")
bermudan_put = price_option(**args, style="bermudan", exercise_dates={15, 30})

print(f"European put:  {euro_put:.4f}")
print(f"American put:  {amer_put:.4f}")
print(f"Bermudan put:  {bermudan_put:.4f}")


In [ ]:
def bs_call(S, K, r, sigma, T):
    cdf = NormalDist().cdf
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * cdf(d1) - K * np.exp(-r * T) * cdf(d2)

S, K, r, sigma, T = 100.0, 100.0, 0.05, 0.20, 1.0
bs = bs_call(S, K, r, sigma, T)

levels = np.arange(5, 201, 5)
errors = [
    abs(price_option(S, K, r, sigma, T, N, kind="call", style="european") - bs)
    for N in levels
]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(levels, errors, label="CRR", marker=".")
ax.set_xlabel("Tree levels N")
ax.set_ylabel("|Price - Black-Scholes|")
ax.set_title("CRR convergence to Black-Scholes")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "tree_convergence.png", dpi=150)
plt.show()
